# Scene completion
The pipeline to complete the partial environment scan.

In [ ]:
import numpy as np
import open3d as o3d
import trimesh
import os
import sys
sys.path.insert(0, '../')
import drm
import drm.detect
import drm.align
from PIL import Image
import torch
import numpy as np
from pathlib import Path

%load_ext autoreload
%autoreload 2

## V-Scan Scene Completion
Perform the scene completion on the V-Scan dataset by:
- Loading the pointcloud
- loading the object Boundingboxes
- loading the scene occlusions
- Cutting out the bounding boxes
- Detecting the planes
- Finding plane intersections
- Isolating the points per plane
- filling in the parts between the intersections and in the occlusion regions
- Painting in the missing textures

### Data Loading

In [ ]:
scene_dir = Path(r"/home/jvermandere/datasets/V-Scan/data/Office_1_Leica-P30_1775812306385")

SCAN_FILE = "main.txt"
PANO_FILE = "pano.png"
BB_FILE = "main_bb.json"
OCCLUSION_FILE = "occluded_grid.ply"
VOXEL_SIZE = 0.05

pcd, matrix = drm.txt_pcd_to_open3d(scene_dir / SCAN_FILE)
#pcd.translate(matrix[:3, 3])
pcd = pcd.voxel_down_sample(VOXEL_SIZE)
translate_matrix = np.eye(4)
translate_matrix[:3, 3] = -matrix[:3, 3]

boxes = drm.trimesh_to_open3d(drm.detect.load_gt_json_boxes_as_mesh(scene_dir / BB_FILE))
expanded_boxes = drm.expand_mesh(boxes, offset=0.1)

occlusion_points = o3d.io.read_point_cloud(scene_dir / OCCLUSION_FILE)
occlusion_grid = drm.load_voxelgrid_from_ply(scene_dir / OCCLUSION_FILE, transform=translate_matrix)
scene = drm.visualise_open3d([pcd] + expanded_boxes + [occlusion_grid], voxel_scale_modifier=0.9)
scene.add_geometry(trimesh.creation.axis(0.1))
scene.show()

### Bounding box cutout

In [ ]:
isolated_pcds, remainder_pcd = drm.detect.split_pointcloud_by_boxes(pcd, expanded_boxes)

scene = drm.visualise_open3d(isolated_pcds, random_color=True)
scene.add_geometry(drm.o3d_pointcloud_to_trimesh(remainder_pcd))
scene.show()

### Plane Detection

In [ ]:
detected_planes, plane_models, leftovers = drm.detect.detect_planes(remainder_pcd,min_points=100, num_iterations=1000, distance_threshold=0.1)

drm.visualise_open3d(detected_planes, random_color=True).show()

### Plane boundary detection

In [ ]:
from typing import Optional 
def create_plane_meshes(
    plane_models: list[np.ndarray],
    bounding_box: o3d.geometry.AxisAlignedBoundingBox,
) -> list[o3d.geometry.TriangleMesh]:
    """
    Create finite plane meshes from RANSAC plane models, clipped to twice the
    bounding box extent, then cut each plane by all other planes to produce
    non-overlapping segments covering the full plane.

    Parameters
    ----------
    plane_models : List of [a, b, c, d] arrays (ax + by + cz + d = 0).
    bounding_box : AABB used to size the initial plane quads.

    Returns
    -------
    List of TriangleMesh fragments — all pieces of all planes after mutual cutting.
    """
    center  = bounding_box.get_center()
    extents = np.asarray(bounding_box.get_extent()) * 2.0
    size    = np.linalg.norm(extents)

    plane_meshes = [_plane_model_to_mesh(m, center, size) for m in plane_models]

    result = []
    for i, mesh in enumerate(plane_meshes):
        # Start with the full quad and recursively split by every other plane
        fragments = [mesh]
        for j, model in enumerate(plane_models):
            if i == j:
                continue
            next_fragments = []
            for fragment in fragments:
                pos, neg = _split_mesh_by_plane(fragment, model)
                if pos is not None and len(pos.triangles) > 0:
                    _orient_normals_toward(pos, center)
                    next_fragments.append(pos)
                if neg is not None and len(neg.triangles) > 0:
                    _orient_normals_toward(neg, center)
                    next_fragments.append(neg)
            fragments = next_fragments

        result.extend(fragments)

    return result

def _orient_normals_toward(
    mesh: o3d.geometry.TriangleMesh,
    target: np.ndarray,
) -> None:
    """
    Flip any triangle whose normal points away from `target` (in-place).
    """
    mesh.compute_triangle_normals()
    verts   = np.asarray(mesh.vertices)
    faces   = np.asarray(mesh.triangles)
    normals = np.asarray(mesh.triangle_normals)

    # Face centroid → vector toward target
    centroids   = verts[faces].mean(axis=1)          # (F, 3)
    to_target   = target - centroids                 # (F, 3)
    facing_away = np.einsum("fd,fd->f", normals, to_target) < 0  # (F,)

    # Flip winding of offending triangles
    faces[facing_away] = faces[facing_away][:, ::-1]
    mesh.triangles = o3d.utility.Vector3iVector(faces)
    mesh.compute_vertex_normals()
    mesh.compute_triangle_normals()

# ── helpers ───────────────────────────────────────────────────────────────────

def _plane_model_to_mesh(
    model: np.ndarray,
    center: np.ndarray,
    size: float,
) -> o3d.geometry.TriangleMesh:
    a, b, c, d = model
    normal = np.array([a, b, c], dtype=np.float64)
    normal /= np.linalg.norm(normal)

    t = -(np.dot(normal, center) + d) / np.dot(normal, normal)
    plane_center = center + t * normal

    up = np.array([0.0, 0.0, 1.0])
    if abs(np.dot(normal, up)) > 0.9:
        up = np.array([0.0, 1.0, 0.0])
    u = np.cross(normal, up);  u /= np.linalg.norm(u)
    v = np.cross(normal, u);   v /= np.linalg.norm(v)

    corners = np.array([
        plane_center + size * (-u - v),
        plane_center + size * ( u - v),
        plane_center + size * ( u + v),
        plane_center + size * (-u + v),
    ])

    mesh = o3d.geometry.TriangleMesh()
    mesh.vertices  = o3d.utility.Vector3dVector(corners)
    mesh.triangles = o3d.utility.Vector3iVector([[0, 1, 2], [0, 2, 3]])
    mesh.compute_vertex_normals()
    return mesh


def _split_mesh_by_plane(
    mesh: o3d.geometry.TriangleMesh,
    model: np.ndarray,
) -> tuple[Optional[o3d.geometry.TriangleMesh], Optional[o3d.geometry.TriangleMesh]]:
    """
    Split a mesh into two halves along a plane.

    Returns
    -------
    pos : fragment where dot(normal, x) + d > 0  (in front of plane)
    neg : fragment where dot(normal, x) + d <= 0 (behind plane)
    """
    a, b, c, d = model
    normal = np.array([a, b, c], dtype=np.float64)
    normal /= np.linalg.norm(normal)

    verts = np.asarray(mesh.vertices)
    faces = np.asarray(mesh.triangles)
    dists = np.dot(verts, normal) + d   # signed distance per vertex

    def intersect(p1, p2, d1, d2):
        t = d1 / (d1 - d2)
        return p1 + t * (p2 - p1)

    pos_verts, pos_faces = [], []
    neg_verts, neg_faces = [], []

    def add_vertex(buf, p):
        buf.append(p)
        return len(buf) - 1

    def clip_polygon(poly, poly_dists, keep_positive: bool):
        """Sutherland-Hodgman for one side."""
        clipped = []
        n = len(poly)
        for k in range(n):
            curr, nxt   = poly[k], poly[(k + 1) % n]
            dc, dn      = poly_dists[k], poly_dists[(k + 1) % n]
            inside_curr = (dc > 0) if keep_positive else (dc <= 0)
            inside_next = (dn > 0) if keep_positive else (dn <= 0)
            if inside_curr:
                clipped.append(curr)
            if inside_curr != inside_next:
                clipped.append(intersect(curr, nxt, dc, dn))
        return clipped

    for tri in faces:
        poly       = [verts[i] for i in tri]
        poly_dists = [dists[i] for i in tri]

        for keep_positive, vert_buf, face_buf in (
            (True,  pos_verts, pos_faces),
            (False, neg_verts, neg_faces),
        ):
            clipped = clip_polygon(poly, poly_dists, keep_positive)
            if len(clipped) >= 3:
                idx = [add_vertex(vert_buf, p) for p in clipped]
                for k in range(1, len(idx) - 1):
                    face_buf.append([idx[0], idx[k], idx[k + 1]])

    def build(vbuf, fbuf):
        if not fbuf:
            return None
        m = o3d.geometry.TriangleMesh()
        m.vertices  = o3d.utility.Vector3dVector(np.array(vbuf))
        m.triangles = o3d.utility.Vector3iVector(np.array(fbuf))
        m.merge_close_vertices(1e-8)
        m.remove_duplicated_vertices()
        m.remove_duplicated_triangles()
        m.compute_vertex_normals()
        return m

    return build(pos_verts, pos_faces), build(neg_verts, neg_faces)

plane_meshes = create_plane_meshes(plane_models, pcd.get_axis_aligned_bounding_box())
print(f"Detected {len(plane_meshes)} planes after clipping.")
scene = drm.visualise_open3d(plane_meshes + [pcd], random_color=True)
scene.show()

In [ ]:
def filter_planes_by_points(
    plane_meshes: list[o3d.geometry.TriangleMesh],
    plane_pcds: list[o3d.geometry.PointCloud],
    distance_threshold: float = 0.01,
    min_points: int = 10,
) -> tuple[list[o3d.geometry.TriangleMesh], list[o3d.geometry.PointCloud]]:
    kept_meshes = []
    kept_pcds   = []

    for mesh in plane_meshes:
        counts = []
        for pcd in plane_pcds:
            count = _count_points_in_mesh_fragment(pcd, mesh, distance_threshold)
            counts.append(count)

        total_count = sum(counts)
        best_plane  = int(np.argmax(counts))

        if total_count >= min_points:
            kept_meshes.append(mesh)
            kept_pcds.append(plane_pcds[best_plane])

    print(f"Kept {len(kept_meshes)}/{len(plane_meshes)} fragments.")
    return kept_meshes, kept_pcds

def _count_points_in_mesh_fragment(
    pcd: o3d.geometry.PointCloud,
    mesh: o3d.geometry.TriangleMesh,
    distance_threshold: float,
) -> int:
    if len(pcd.points) == 0 or len(mesh.triangles) == 0:
        return 0

    points = np.asarray(pcd.points)

    # Step 1: distance filter — keep points close to the plane surface
    mesh_pcd = mesh.sample_points_uniformly(number_of_points=2000)
    distances = np.asarray(pcd.compute_point_cloud_distance(mesh_pcd))
    near_mask = distances <= distance_threshold
    if not np.any(near_mask):
        return 0

    near_points = points[near_mask]

    # Step 2: project onto the plane and test inside the fragment's 2D boundary
    inside_mask = _points_in_mesh_2d(near_points, mesh)

    return int(np.sum(inside_mask))


def _points_in_mesh_2d(
    points: np.ndarray,
    mesh: o3d.geometry.TriangleMesh,
) -> np.ndarray:
    """
    Test whether points lie inside a flat mesh by projecting everything onto
    the mesh plane and doing a 2D point-in-triangles test.

    Works for any mesh orientation — projects into the mesh's local UV frame.
    """
    mesh.compute_triangle_normals()
    verts   = np.asarray(mesh.vertices)
    faces   = np.asarray(mesh.triangles)
    normal  = np.asarray(mesh.triangle_normals).mean(axis=0)
    normal /= np.linalg.norm(normal)

    # Build a local 2D frame (u, v) in the plane
    up = np.array([0.0, 0.0, 1.0])
    if abs(np.dot(normal, up)) > 0.9:
        up = np.array([0.0, 1.0, 0.0])
    u = np.cross(normal, up);  u /= np.linalg.norm(u)
    v = np.cross(normal, u);   v /= np.linalg.norm(v)

    # Project vertices and query points onto the 2D frame
    origin    = verts.mean(axis=0)
    verts_2d  = np.stack([np.dot(verts  - origin, u),
                          np.dot(verts  - origin, v)], axis=1)
    points_2d = np.stack([np.dot(points - origin, u),
                          np.dot(points - origin, v)], axis=1)

    # Test each point against all triangles
    inside = np.zeros(len(points), dtype=bool)
    for tri in faces:
        a, b, c = verts_2d[tri[0]], verts_2d[tri[1]], verts_2d[tri[2]]
        inside |= _points_in_triangle_2d(points_2d, a, b, c)

    return inside


def _points_in_triangle_2d(
    points: np.ndarray,
    a: np.ndarray,
    b: np.ndarray,
    c: np.ndarray,
) -> np.ndarray:
    """
    Barycentric test: True where each point lies inside triangle (a, b, c).
    """
    v0 = c - a
    v1 = b - a
    v2 = points - a

    dot00 = np.dot(v0, v0)
    dot01 = np.dot(v0, v1)
    dot11 = np.dot(v1, v1)
    dot02 = v2 @ v0
    dot12 = v2 @ v1

    denom = dot00 * dot11 - dot01 * dot01
    if abs(denom) < 1e-10:
        return np.zeros(len(points), dtype=bool)

    inv = 1.0 / denom
    u_coord = (dot11 * dot02 - dot01 * dot12) * inv
    v_coord = (dot00 * dot12 - dot01 * dot02) * inv

    return (u_coord >= 0) & (v_coord >= 0) & (u_coord + v_coord <= 1)

filtered_meshes, filtered_pcds = filter_planes_by_points(plane_meshes, detected_planes, distance_threshold=0.1, min_points=100)

scene = drm.visualise_open3d(filtered_meshes +  filtered_pcds, random_color=True)
scene.add_geometry(trimesh.creation.axis(0.1))
scene.show()

In [ ]:
def sample_unoccupied_plane_points(
    plane_meshes: list[o3d.geometry.TriangleMesh],
    plane_pcds: list[o3d.geometry.PointCloud],
) -> list[o3d.geometry.PointCloud]:
    """
    For each plane fragment, sample points at the same voxel density as its
    corresponding point cloud, then keep only the sampled points that are
    NOT within one voxel size of any existing point.

    Parameters
    ----------
    plane_meshes : Filtered plane mesh fragments.
    plane_pcds   : Corresponding segmented point clouds, same order.

    Returns
    -------
    List of PointClouds containing only the unoccupied sampled points.
    """
    result = []

    for mesh, pcd in zip(plane_meshes, plane_pcds):
        pcd_points = np.asarray(pcd.points)
        n_pcd      = len(pcd_points)

        if n_pcd < 3:
            result.append(o3d.geometry.PointCloud())
            continue

        # --- estimate voxel size from median nn distance ---
        nn_distances = np.asarray(pcd.compute_nearest_neighbor_distance())
        voxel_size   = float(np.median(nn_distances))

        # --- sample mesh surface densely then voxel downsample ---
        # oversample so voxel grid is well populated
        n_oversample = max(int(mesh.get_surface_area() / (voxel_size ** 2)) * 4, 1000)
        sampled      = mesh.sample_points_uniformly(number_of_points=n_oversample)
        sampled_down = sampled.voxel_down_sample(voxel_size)

        # --- keep only sampled points far from existing pcd points ---
        distances       = np.asarray(sampled_down.compute_point_cloud_distance(pcd))
        unoccupied_mask = distances > voxel_size

        unoccupied_pts = np.asarray(sampled_down.points)[unoccupied_mask]

        unoccupied_pcd        = o3d.geometry.PointCloud()
        unoccupied_pcd.points = o3d.utility.Vector3dVector(unoccupied_pts)

        result.append(unoccupied_pcd)
        print(f"Plane: {n_pcd} existing, {len(np.asarray(sampled_down.points))} voxel-sampled, "
              f"{len(unoccupied_pts)} unoccupied (voxel_size={voxel_size:.4f})")

    return result

unoccupied = sample_unoccupied_plane_points(filtered_meshes, filtered_pcds)

# Visualise occupied vs unoccupied side by side
scene = drm.visualise_open3d(filtered_pcds, random_color=True)
for pcd in unoccupied:
    tm = drm.o3d_pointcloud_to_trimesh(pcd)
    tm.colors = np.full((len(tm.vertices), 4), [255, 50, 50, 255], dtype=np.uint8)
    scene.add_geometry(tm)
scene.show()

In [ ]:
def plane_pointclouds_to_meshes(
    plane_pcds: list[o3d.geometry.PointCloud],
    plane_meshes: list[o3d.geometry.TriangleMesh],
) -> list[o3d.geometry.TriangleMesh]:
    """
    For each plane fragment:
      1. Subdivide the fragment mesh to match the pcd voxel density.
      2. Project each subdivided vertex onto the pcd surface by displacing
         it along the plane normal to the closest pcd point.

    Parameters
    ----------
    plane_pcds   : Segmented plane point clouds.
    plane_meshes : Corresponding filtered plane mesh fragments.

    Returns
    -------
    List of TriangleMesh with vertices projected onto the pcd surface.
    """
    from scipy.spatial import cKDTree

    result = []

    for pcd, mesh in zip(plane_pcds, plane_meshes):
        points = np.asarray(pcd.points)

        if len(points) < 3 or len(mesh.triangles) == 0:
            result.append(o3d.geometry.TriangleMesh())
            continue

        # --- voxel size from pcd point spacing ---
        voxel_size = float(np.median(
            np.asarray(pcd.compute_nearest_neighbor_distance())
        ))

        # --- fit plane normal via SVD ---
        centroid  = points.mean(axis=0)
        _, _, vh  = np.linalg.svd(points - centroid)
        normal    = vh[2];  normal /= np.linalg.norm(normal)

        # --- subdivide mesh to voxel density ---
        subdiv_mesh = _subdivide_mesh_to_density(mesh, voxel_size)

        # --- project vertices onto pcd surface along normal ---
        verts = np.asarray(subdiv_mesh.vertices).copy()
        tree  = cKDTree(points)

        _, nn_idx    = tree.query(verts, k=1)
        closest_pts  = points[nn_idx]               # (V, 3)

        # Vector from each vertex to its closest pcd point
        delta = closest_pts - verts                 # (V, 3)

        # Project delta onto the plane normal to get perpendicular displacement
        perp_dist = np.einsum("vd,d->v", delta, normal)   # (V,) signed distances
        displaced = verts + perp_dist[:, None] * normal    # move along normal only

        subdiv_mesh.vertices = o3d.utility.Vector3dVector(displaced)
        subdiv_mesh.compute_vertex_normals()
        subdiv_mesh.compute_triangle_normals()
        subdiv_mesh.orient_triangles()

        result.append(subdiv_mesh)
        print(f"Projected mesh: {len(displaced)} verts, "
              f"{len(np.asarray(subdiv_mesh.triangles))} tris, "
              f"voxel_size={voxel_size:.4f}")

    return result


def _subdivide_mesh_to_density(
    mesh: o3d.geometry.TriangleMesh,
    target_edge_length: float,
) -> o3d.geometry.TriangleMesh:
    """
    Subdivide a mesh until no edge is longer than `target_edge_length`.
    Uses midpoint subdivision iteratively.
    """
    mesh = o3d.geometry.TriangleMesh(mesh)   # copy

    for _ in range(10):                      # max iterations as safety cap
        verts = np.asarray(mesh.vertices)
        faces = np.asarray(mesh.triangles)

        # Compute max edge length across all triangles
        e0 = np.linalg.norm(verts[faces[:, 1]] - verts[faces[:, 0]], axis=1)
        e1 = np.linalg.norm(verts[faces[:, 2]] - verts[faces[:, 1]], axis=1)
        e2 = np.linalg.norm(verts[faces[:, 0]] - verts[faces[:, 2]], axis=1)
        max_edge = max(e0.max(), e1.max(), e2.max())

        if max_edge <= target_edge_length:
            break

        mesh = mesh.subdivide_midpoint(number_of_iterations=1)

    return mesh

projected_meshes = plane_pointclouds_to_meshes(filtered_pcds, filtered_meshes)
scene = drm.visualise_open3d(projected_meshes, random_color=True)
scene.show()

## Old Scene Completion

## Load pointcloud

In [ ]:
pcdPath = "/home/jvermandere/projects/DRM/_input/virtualDataset/VirtualScanner-1773153754053/results/segmented_points/isolated_points.txt"
pointsColors = np.loadtxt(pcdPath, dtype=np.float32).reshape(-1, 6)
cloud = trimesh.points.PointCloud(pointsColors[:, :3], colors=pointsColors[:, 3:6]/255)
scene = trimesh.Scene(cloud)
scene.show()

## Ransac itterative plane detection

In [ ]:
min_points = 100  # minimum points to consider a plane
remaining_points = cloud.vertices.copy()
remaining_colors = cloud.colors.copy() if cloud.colors is not None else None

planes = []  # list of trimesh point clouds for each plane

while len(remaining_points) >= min_points:
    plane_model, inliers = drm.detect.ransac_plane_trimesh(remaining_points,
                                        num_iterations=1000,
                                        distance_threshold=0.01)

    if len(inliers) < min_points:
        print("No more large planes detected.")
        break

    # Extract plane points and colors
    plane_pts = remaining_points[inliers]
    plane_colors = remaining_colors[inliers] if remaining_colors is not None else None
    plane_pc = trimesh.points.PointCloud(vertices=plane_pts, colors=plane_colors)
    planes.append(plane_pc)

    # Remove plane points from remaining points
    mask = np.ones(len(remaining_points), dtype=bool)
    mask[inliers] = False
    remaining_points = remaining_points[mask]
    if remaining_colors is not None:
        remaining_colors = remaining_colors[mask]

# Remaining points as a separate point cloud
if len(remaining_points) > 0:
    remaining_pc = trimesh.points.PointCloud(vertices=remaining_points,
                                             colors=remaining_colors)
else:
    remaining_pc = None

print(f"Extracted {len(planes)} planes.")

In [ ]:
scene = drm.visualize_pointclouds_random_colors(planes)
scene.show()

## Inpainting all the planes

In [ ]:
from simple_lama_inpainting import SimpleLama
simple_lama = SimpleLama()
# Paint in all the planes one by one
filled_planes = []
for i in range(len(planes)):
    #fill in the planes
    filled_plane = drm.fill_plane_holes(planes[i], target_density=0.01)
    # get the 2D images of the partial plane and to-be-filled-in area of the new plane
    img, mask, filled_coords = drm.project_planes_with_infill_mask(planes[i], filled_plane, resolution=512, point_radius=1)
    # paint in the image
    image_pil = Image.fromarray(img.astype(np.uint8)).convert("RGB").resize((512, 512))
    mask_pil = Image.fromarray((mask * 255).astype(np.uint8)).convert("L").resize((512, 512))
    # Run inpainting
    inpaintedImage = simple_lama(image_pil,mask_pil)
    #
    texture = np.asarray(inpaintedImage)
    colors = texture[filled_coords[:,1], filled_coords[:,0]]
    filled_plane.colors = np.hstack([colors,np.full((len(colors),1),255)])
    filled_planes.append(filled_plane)


In [ ]:
scene = trimesh.Scene(filled_planes[:4])
scene.show()

### Save the completed pointcloud

In [ ]:
all_vertices = []
all_colors = []

for pc in filled_planes[:4]:
    all_vertices.append(pc.vertices)

    if pc.colors is not None:
        all_colors.append(pc.colors)
    else:
        # default white
        all_colors.append(np.full((len(pc.vertices), 4), 255, dtype=np.uint8))

vertices = np.vstack(all_vertices)
colors = np.vstack(all_colors)

merged_pc = trimesh.points.PointCloud(vertices=vertices, colors=colors)

merged_pc.export(Path(pcdPath).as_posix()[:-4] + "_reconstructed.ply")

## Inpainting one plane

In [ ]:
filled_planes = []
for plane in planes:
    filled_plane = drm.fill_plane_holes(plane, target_density=0.01)
    filled_planes.append(filled_plane)

scene = trimesh.Scene(filled_planes[:4])
scene.show()

In [ ]:
import matplotlib.pyplot as plt

# filled_plane = fill_plane_holes_with_colors(plane)
img, mask, filled_coords = drm.project_planes_with_infill_mask(planes[3], filled_planes[3], resolution=512, point_radius=1)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
plt.imshow(img)
plt.title("Original Plane Colors")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(mask, cmap='gray')
plt.title("Filled Areas / Holes")
plt.axis("off")

plt.show()

### StableDiffusion Inpainting

In [ ]:
from PIL import Image
import torch
from diffusers import StableDiffusionInpaintPipeline
import numpy as np
# Load inpainting model
pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting"
).to("cuda")



In [ ]:

# Convert numpy arrays
image_pil = Image.fromarray(img.astype(np.uint8)).convert("RGB")
mask_pil = Image.fromarray((mask * 255).astype(np.uint8)).convert("L")

# Resize both to same dimensions (512x512 is safe for SD)
image_pil = image_pil.resize((512, 512))
mask_pil = mask_pil.resize((512, 512))



#prompt = "paint in the masked area to fill in the missing regions looking at the rest of the colors in the image to match the texture, ignore eveything that is black"
prompt = "paint"

# Run inpainting
result = pipe(
    prompt=prompt,
    image=image_pil,
    mask_image=mask_pil
).images[0]


In [ ]:
result

### LaMa

In [ ]:
from simple_lama_inpainting import SimpleLama
from PIL import Image

simple_lama = SimpleLama()

# Convert numpy arrays
image_pil = Image.fromarray(img.astype(np.uint8)).convert("RGB")
mask_pil = Image.fromarray((mask * 255).astype(np.uint8)).convert("L")
image_pil = image_pil.resize((512, 512))
mask_pil = mask_pil.resize((512, 512))

result = simple_lama(image_pil, mask_pil)
result

### Reprojection

In [ ]:
def apply_texture_to_plane(filled_pc, inpainted_img, filled_coords):

    img = np.asarray(inpainted_img)

    new_colors = np.zeros((len(filled_coords), 4), dtype=np.uint8)

    for i, (x,y) in enumerate(filled_coords):

        color = img[y, x]

        if len(color) == 3:
            new_colors[i] = np.append(color, 255)
        else:
            new_colors[i] = color

    filled_pc.colors = new_colors

    return filled_pc

In [ ]:
filled_pc = apply_texture_to_plane(
    filled_planes[3],
    result,
    filled_coords
)

In [ ]:
scene = trimesh.Scene(filled_pc)
scene.show()

In [ ]:
from simple_lama_inpainting import SimpleLama
from PIL import Image

simple_lama = SimpleLama()

image_pil = image_pil.resize((512, 512))
mask_pil = mask_pil.resize((512, 512))

result = simple_lama(image_pil, mask_pil)
result